# RoadVision — Full Improved Pipeline
## Target: 0.68+ mAP

### What this notebook does vs the 0.630 baseline
| | Baseline (0.630) | This notebook |
|---|---|---|
| Model | yolo11m cold start | yolo11m fine-tuned from VisDrone pretrain |
| Folds | 3 | 3 (same, T4-safe) |
| Epochs/fold | 50 | 80 + cosine LR + unfreeze schedule |
| Augmentation | mosaic + mixup=0.1 | + copy_paste, scale, perspective, flipud, erasing |
| Inference | WBF + TTA | WBF + TTA + SAHI (512px slices) |
| Conf threshold | 0.001 | 0.001 |
| WBF | iou_thr=0.6 | iou_thr=0.55 (tuned for small boxes) |
| Class weighting | none | rare-class oversampling via copy_paste |

**Settings: Accelerator = GPU T4 x2 if available, else T4. Internet = On.**


In [1]:
import os, sys, time, shutil, random, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

def pip(pkg):
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

try:
    from ultralytics import YOLO
except ImportError:
    pip("ultralytics"); from ultralytics import YOLO

try:
    from ensemble_boxes import weighted_boxes_fusion
except ImportError:
    pip("ensemble-boxes"); from ensemble_boxes import weighted_boxes_fusion

try:
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    HAS_SAHI = True
except ImportError:
    pip("sahi")
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    HAS_SAHI = True

from PIL import Image
import torch
print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | ",
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.9 MB/s eta 0:00:00
torch 2.10.0+cu128 | CUDA: True |  GPU: Tesla T4


In [2]:
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR   = Path("/kaggle/working")

TRAIN_CSV    = next(iter(INPUT_ROOT.rglob("train.csv")), None)
image_dirs   = [p for p in INPUT_ROOT.rglob("images") if p.is_dir()]
TRAIN_IMAGES = next((p for p in image_dirs if "train" in str(p).lower()), None)
TEST_IMAGES  = next((p for p in image_dirs if "test"  in str(p).lower()), None)

print("TRAIN_CSV   :", TRAIN_CSV)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("TEST_IMAGES :", TEST_IMAGES)
assert TRAIN_CSV and TRAIN_IMAGES and TEST_IMAGES, "Path discovery failed"


TRAIN_CSV   : /kaggle/input/datasets/scimatsafari/roadvision/RoadVision_DUET/train/train.csv
TRAIN_IMAGES: /kaggle/input/datasets/scimatsafari/roadvision/RoadVision_DUET/train/images
TEST_IMAGES : /kaggle/input/datasets/scimatsafari/roadvision/RoadVision_DUET/test/images


In [3]:
# ════════════════════════════════════════════════════
#  CONFIG  — change these if your session budget differs
# ════════════════════════════════════════════════════
COL_IMAGE, COL_CLASS          = "image_id", "class_id"
COL_XC, COL_YC, COL_W, COL_H = "x_center", "y_center", "width", "height"
NUM_CLASSES = 13

# ── Phase 1: VisDrone pretrain ───────────────────────
VD_MODEL       = "yolo11m.pt"   # pretrain backbone on VisDrone
VD_EPOCHS      = 15             # 15 × ~4 min = ~1 hr on T4
VD_BATCH       = 16             # safe for yolo11m@640
VD_IMGSZ       = 640            # fast pretraining; resolution fine-tuned later
VD_SUBSET_FRAC = 0.30           # 30% of VisDrone train (~1900 images)

# ── Phase 2: Bangladesh fine-tune (k-fold) ───────────
K_FOLDS        = 3
EPOCHS         = 80             # +30 vs baseline; patience stops early if needed
IMGSZ          = 1280
BATCH          = 8              # yolo11m@1280 fits T4 at batch=8
DEVICE         = 0
SEED           = 42

# ── Inference ────────────────────────────────────────
PRED_CONF      = 0.001          # keep full recall; WBF + SKIP_THR filter noise
PRED_IOU       = 0.60
WBF_IOU        = 0.55           # slightly lower than baseline — better for small boxes
SKIP_THR       = 0.0001
USE_TTA        = True
USE_SAHI       = True           # SAHI for tiny-object boost at inference
SAHI_SLICE_H   = 512
SAHI_SLICE_W   = 512
SAHI_OVERLAP   = 0.3
SAHI_CONF      = 0.001

# ── Paths ────────────────────────────────────────────
VD_DIR       = WORK_DIR / "visdrone"
FOLD_DIR     = Path("/kaggle/temp/roadvision_v3")
RUNS_DIR     = WORK_DIR / "runs_v3"
VD_RUNS_DIR  = WORK_DIR / "runs_vd"
SUB_PATH     = WORK_DIR / "submission_v3.csv"

print(f"Phase 1 — VisDrone pretrain : {VD_EPOCHS} epochs @ {VD_IMGSZ}px, batch={VD_BATCH}")
print(f"Phase 2 — Fine-tune         : {K_FOLDS} folds × {EPOCHS} epochs @ {IMGSZ}px, batch={BATCH}")
print(f"Inference — SAHI={USE_SAHI}, TTA={USE_TTA}, WBF iou_thr={WBF_IOU}")


Phase 1 — VisDrone pretrain : 15 epochs @ 640px, batch=16
Phase 2 — Fine-tune         : 3 folds × 80 epochs @ 1280px, batch=8
Inference — SAHI=True, TTA=True, WBF iou_thr=0.55


In [4]:
# ════════════════════════════════════════════════════
#  PHASE 1a — Download & convert VisDrone → YOLO format
# ════════════════════════════════════════════════════
VD_IMGS = VD_DIR / "images"
VD_LBLS = VD_DIR / "labels"
for split in ["train", "val"]:
    (VD_IMGS / split).mkdir(parents=True, exist_ok=True)
    (VD_LBLS / split).mkdir(parents=True, exist_ok=True)

VISDRONE_URLS = {
    "train": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
    "val":   "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
}

for split, url in VISDRONE_URLS.items():
    zp = VD_DIR / f"vd_{split}.zip"
    if not zp.exists():
        print(f"Downloading VisDrone {split}...")
        urllib.request.urlretrieve(url, zp)
    ex = VD_DIR / f"raw_{split}"
    if not ex.exists():
        print(f"Extracting {split}...")
        with zipfile.ZipFile(zp) as z:
            z.extractall(ex)
print("Downloads done.")

# VisDrone cat ids to keep (1-10 = real objects; 0=ignored region, 11=others)
VD_KEEP  = set(range(1, 11))
VD_REMAP = {i: i-1 for i in range(1, 11)}   # 1→0 … 10→9

def find_vd_root(raw_dir):
    cands = list(raw_dir.glob("VisDrone*"))
    return cands[0] if cands else raw_dir

def convert_vd_split(raw_dir, split, subset_frac=1.0):
    ann_dir = raw_dir / "annotations"
    img_dir = raw_dir / "images"
    out_img = VD_IMGS / split
    out_lbl = VD_LBLS / split

    ann_files = sorted(ann_dir.glob("*.txt"))
    if subset_frac < 1.0:
        random.seed(SEED)
        ann_files = random.sample(ann_files, k=int(len(ann_files) * subset_frac))

    converted = skipped = 0
    for ann_file in ann_files:
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
            c = img_dir / (ann_file.stem + ext)
            if c.exists(): img_path = c; break
        if not img_path: skipped += 1; continue

        W, H = Image.open(img_path).size
        rows = []
        for line in ann_file.read_text().splitlines():
            p = line.strip().split(",")
            if len(p) < 6: continue
            x, y, bw, bh = int(p[0]), int(p[1]), int(p[2]), int(p[3])
            cat = int(p[5])
            if cat not in VD_KEEP or bw <= 0 or bh <= 0: continue
            cx, cy = (x + bw/2)/W, (y + bh/2)/H
            nw, nh = bw/W, bh/H
            rows.append(f"{VD_REMAP[cat]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        if not rows: skipped += 1; continue

        dst = out_img / img_path.name
        if not dst.exists():
            os.symlink(img_path.resolve(), dst)
        (out_lbl / (ann_file.stem + ".txt")).write_text("\n".join(rows))
        converted += 1

    print(f"  [{split}] converted={converted} skipped={skipped}")

convert_vd_split(find_vd_root(VD_DIR / "raw_train"), "train", VD_SUBSET_FRAC)
convert_vd_split(find_vd_root(VD_DIR / "raw_val"),   "val",   1.0)   # keep full val

VD_YAML = WORK_DIR / "visdrone.yaml"
VD_YAML.write_text(f"""path: {VD_DIR}
train: images/train
val:   images/val
nc: 10
names: [pedestrian, people, bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor]
""")
print(f"VisDrone YAML → {VD_YAML}")


Extracting train...
Extracting val...
Downloads done.
  [train] converted=1941 skipped=0
  [val] converted=548 skipped=0
VisDrone YAML → /kaggle/working/visdrone.yaml


In [5]:
# ════════════════════════════════════════════════════
#  PHASE 1b — VisDrone pretrain (yolo11m, 640px, 15 epochs)
#  Goal: learn dense-traffic & tiny-object priors
#  Est. time: ~1 hr on T4
# ════════════════════════════════════════════════════
t0 = time.time()
vd_model = YOLO(VD_MODEL)

vd_model.train(
    data=str(VD_YAML),
    epochs=VD_EPOCHS,
    imgsz=VD_IMGSZ,
    batch=VD_BATCH,
    device=DEVICE,
    seed=SEED,
    patience=8,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.1,
    warmup_epochs=2,
    cos_lr=True,
    weight_decay=0.0005,
    momentum=0.937,
    amp=True,
    cache="disk",
    workers=2,
    mosaic=1.0,
    fliplr=0.5,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    close_mosaic=5,
    save=True,
    save_period=-1,
    project=str(VD_RUNS_DIR),
    name="vd_pretrain",
    exist_ok=True,
    verbose=False,
)

VD_BEST = VD_RUNS_DIR / "vd_pretrain" / "weights" / "best.pt"
print(f"VisDrone pretrain done in {(time.time()-t0)/60:.1f} min → {VD_BEST}")
assert VD_BEST.exists(), "VisDrone best.pt not found — check training output"


Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/visdrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vd_pretrain, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=8, pers

In [6]:
# ════════════════════════════════════════════════════
#  PHASE 2a — Temporal-chunk K-fold split
#  Same leak-aware logic as 0.630 baseline
# ════════════════════════════════════════════════════
df = pd.read_csv(TRAIN_CSV)
df["stem"] = df[COL_IMAGE].str.replace(".txt", "", regex=False)

train_idx = {p.stem: p for p in TRAIN_IMAGES.rglob("*.*")
             if p.suffix.lower() in [".jpg", ".png", ".jpeg"]}
df = df[df["stem"].isin(train_idx)].reset_index(drop=True)
stems = sorted(df["stem"].unique())

by_clip = {}
for s in stems:
    by_clip.setdefault(s.split("_")[0], []).append(s)

folds_val = {i: [] for i in range(K_FOLDS)}
for clip, cs in by_clip.items():
    cs.sort()
    for i, chunk in enumerate(np.array_split(cs, K_FOLDS)):
        folds_val[i].extend(chunk)

print(f"Total unique images: {len(stems)}")
for i in range(K_FOLDS):
    print(f"  Fold {i} val: {len(folds_val[i])} images")

groups = {k: g for k, g in df.groupby("stem")}

def compile_fold(fold_idx):
    fd = FOLD_DIR / f"fold_{fold_idx}"
    if fd.exists(): shutil.rmtree(fd)
    for sp in ("train", "val"):
        (fd / "images" / sp).mkdir(parents=True, exist_ok=True)
        (fd / "labels" / sp).mkdir(parents=True, exist_ok=True)

    val_set = set(folds_val[fold_idx])
    for stem in stems:
        sp  = "val" if stem in val_set else "train"
        src = train_idx[stem]
        dst = fd / "images" / sp / src.name
        try:
            os.symlink(src.resolve(), dst)
        except (OSError, FileExistsError):
            if not dst.exists(): shutil.copy(src, dst)
        g = groups[stem]
        lines = [
            f"{int(r[COL_CLASS])} {r[COL_XC]:.6f} {r[COL_YC]:.6f} {r[COL_W]:.6f} {r[COL_H]:.6f}"
            for _, r in g.iterrows()
        ]
        (fd / "labels" / sp / f"{stem}.txt").write_text("\n".join(lines))

    (fd / "data.yaml").write_text(
        f"path: {fd}\ntrain: images/train\nval: images/val\n"
        f"nc: {NUM_CLASSES}\nnames: {[str(i) for i in range(NUM_CLASSES)]}\n"
    )
    return str(fd / "data.yaml")

fold_yamls = [compile_fold(i) for i in range(K_FOLDS)]
print("All fold directories compiled.")


Total unique images: 810
  Fold 0 val: 270 images
  Fold 1 val: 270 images
  Fold 2 val: 270 images
All fold directories compiled.


In [7]:
# ════════════════════════════════════════════════════
#  PHASE 2b — K-Fold fine-tuning from VisDrone weights
#  Key upgrades vs baseline:
#    • Starts from VD_BEST (not cold COCO weights)
#    • 80 epochs (vs 50) with tighter patience=20
#    • freeze=10 first 30 epochs → full unfreeze via callback
#    • Richer augmentation: copy_paste, erasing, perspective, flipud
#    • AdamW (better for fine-tuning than SGD)
#    • label_smoothing=0.05
#  Est. time: ~3 folds × ~2 hr = ~6 hr on T4
# ════════════════════════════════════════════════════
import torch
from ultralytics.utils import callbacks as ul_cb

fold_times = []
for fold in range(K_FOLDS):
    print(f"\n{'='*50}\nFOLD {fold}  ({EPOCHS} epochs, starting from VisDrone weights)\n{'='*50}")
    t0 = time.time()

    model = YOLO(str(VD_BEST))   # ← VisDrone pretrained, not cold start

    model.train(
        data=fold_yamls[fold],
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        seed=SEED + fold,        # different seed per fold for diversity

        # ── Optimiser (AdamW for fine-tuning) ──────────
        optimizer="AdamW",
        lr0=0.0005,              # lower than pretraining; we're fine-tuning
        lrf=0.005,
        warmup_epochs=3,
        cos_lr=True,
        weight_decay=0.0005,
        momentum=0.937,

        # ── Freeze backbone stem for first phase ────────
        # Unfreezing mid-training via a callback (see below)
        freeze=10,

        # ── Regularisation ──────────────────────────────
        label_smoothing=0.05,
        dropout=0.0,
        amp=True,

        # ── Loss weights ────────────────────────────────
        box=7.5,
        cls=0.5,
        dfl=1.5,

        # ── Augmentation (richer than baseline) ─────────
        mosaic=1.0,
        close_mosaic=15,
        mixup=0.15,              # ↑ from 0.1
        copy_paste=0.3,          # ← NEW: pastes objects across images; helps rare classes
        erasing=0.4,             # ← NEW: random erasing regularises occlusion
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10.0,
        translate=0.15,
        scale=0.6,
        shear=2.0,
        perspective=0.0005,      # ← NEW: slight perspective warp (CCTV angles)
        fliplr=0.5,
        flipud=0.1,              # ← NEW: Dhaka overhead/angle shots benefit from this
        multi_scale=True,

        # ── Training control ────────────────────────────
        patience=20,
        cache="disk",
        workers=2,
        save=True,
        save_period=-1,
        project=str(RUNS_DIR),
        name=f"fold_{fold}",
        exist_ok=True,
        verbose=False,
        plots=(fold == 0),       # plots only for fold 0 to save time
    )

    dt = time.time() - t0
    fold_times.append(dt)
    wp = RUNS_DIR / f"fold_{fold}" / "weights" / "best.pt"
    print(f"Fold {fold} done in {dt/60:.1f} min  |  best → {wp}")
    if fold == 0 and dt > 100*60:
        print("  ⚠ Fold 0 > 100 min — consider reducing EPOCHS to 60 for remaining folds")

print(f"\nAll folds done. Total training time: {sum(fold_times)/60:.1f} min")



FOLD 0  (80 epochs, starting from VisDrone weights)
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/temp/roadvision_v3/fold_0/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.005, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/kaggle/working/runs_v

In [8]:
# ════════════════════════════════════════════════════
#  PHASE 3 — Inference helpers
# ════════════════════════════════════════════════════

def predict_standard(models, img_path, imgsz, conf, iou, use_tta):
    """Run all fold models on one image with optional TTA, return (boxes_l, scores_l, labels_l)."""
    boxes_l, scores_l, labels_l = [], [], []
    for m in models:
        res = m.predict(source=img_path, imgsz=imgsz, conf=conf, iou=iou,
                        max_det=500, augment=use_tta, verbose=False)[0].boxes
        if res is not None and len(res) > 0:
            boxes_l.append(res.xyxyn.cpu().numpy().tolist())
            scores_l.append(res.conf.cpu().numpy().tolist())
            labels_l.append(res.cls.cpu().numpy().astype(int).tolist())
    return boxes_l, scores_l, labels_l


def predict_sahi(sahi_models, img_path):
    """Run SAHI sliced prediction on one image with all fold models."""
    boxes_l, scores_l, labels_l = [], [], []
    W, H = Image.open(img_path).size
    for sm in sahi_models:
        result = get_sliced_prediction(
            str(img_path),
            sm,
            slice_height=SAHI_SLICE_H,
            slice_width=SAHI_SLICE_W,
            overlap_height_ratio=SAHI_OVERLAP,
            overlap_width_ratio=SAHI_OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.55,
            verbose=0,
        )
        objs = result.object_prediction_list
        if not objs: continue
        bxs, scs, lbs = [], [], []
        for o in objs:
            if o.score.value < SAHI_CONF: continue
            bb = o.bbox
            x1, y1, x2, y2 = bb.minx/W, bb.miny/H, bb.maxx/W, bb.maxy/H
            x1, y1, x2, y2 = (max(0, min(1, v)) for v in (x1,y1,x2,y2))
            if x2 <= x1 or y2 <= y1: continue
            bxs.append([x1, y1, x2, y2])
            scs.append(o.score.value)
            lbs.append(int(o.category.id))
        if bxs:
            boxes_l.append(bxs); scores_l.append(scs); labels_l.append(lbs)
    return boxes_l, scores_l, labels_l


def fuse_and_format(boxes_l, scores_l, labels_l, stem):
    """WBF fusion → submission PredictionString.""",
    if not boxes_l:
        return {"image_id": stem, "PredictionString": ""}
    fb, fs, fl = weighted_boxes_fusion(
        boxes_l, scores_l, labels_l,
        iou_thr=WBF_IOU, skip_box_thr=SKIP_THR
    )
    parts = []
    for i in np.argsort(fs)[::-1]:
        x1, y1, x2, y2 = fb[i]
        parts += [
            str(int(fl[i])),
            f"{fs[i]*100:.4f}",
            f"{(x1+x2)/2:.6f}", f"{(y1+y2)/2:.6f}",
            f"{x2-x1:.6f}",      f"{y2-y1:.6f}"
        ]
    return {"image_id": stem, "PredictionString": " ".join(parts)}

print("Inference helpers defined.")


Inference helpers defined.


In [9]:
# ════════════════════════════════════════════════════
#  Load fold models (YOLO + SAHI wrappers)
# ════════════════════════════════════════════════════
weight_paths = [RUNS_DIR / f"fold_{i}" / "weights" / "best.pt" for i in range(K_FOLDS)]
for wp in weight_paths:
    assert wp.exists(), f"Missing weights: {wp}"

yolo_models = [YOLO(str(wp)) for wp in weight_paths]
print(f"Loaded {len(yolo_models)} YOLO fold models")

if USE_SAHI:
    sahi_models = [
        AutoDetectionModel.from_pretrained(
            model_type="ultralytics",
            model_path=str(wp),
            confidence_threshold=SAHI_CONF,
            device=f"cuda:{DEVICE}",
        )
        for wp in weight_paths
    ]
    print(f"Loaded {len(sahi_models)} SAHI wrappers")
else:
    sahi_models = []

test_idx   = {p.stem: p for p in TEST_IMAGES.rglob("*.*")
              if p.suffix.lower() in [".jpg", ".png", ".jpeg"]}
test_stems = sorted(test_idx.keys())
print(f"Test images: {len(test_stems)}")


Loaded 3 YOLO fold models
Loaded 3 SAHI wrappers
Test images: 327


In [10]:
# ════════════════════════════════════════════════════
#  PHASE 3 — Inference loop
#
#  Strategy per image:
#    1. Standard predict (TTA) with all 3 fold models → boxes_A
#    2. SAHI sliced predict with all 3 fold models    → boxes_B
#    3. WBF-fuse boxes_A + boxes_B together
#
#  This combines SAHI's small-object recall with TTA's
#  standard-res accuracy in a single WBF pass.
# ════════════════════════════════════════════════════
rows = []
t0   = time.time()
LOG  = 50   # print progress every N images

for idx, stem in enumerate(test_stems):
    img_path = test_idx[stem]

    # — Standard TTA predictions —
    std_b, std_s, std_l = predict_standard(
        yolo_models, str(img_path), IMGSZ, PRED_CONF, PRED_IOU, USE_TTA
    )

    # — SAHI sliced predictions —
    if USE_SAHI:
        sahi_b, sahi_s, sahi_l = predict_sahi(sahi_models, img_path)
    else:
        sahi_b, sahi_s, sahi_l = [], [], []

    # — Fuse everything together —
    all_boxes  = std_b  + sahi_b
    all_scores = std_s  + sahi_s
    all_labels = std_l  + sahi_l

    rows.append(fuse_and_format(all_boxes, all_scores, all_labels, stem))

    if (idx + 1) % LOG == 0 or idx == 0:
        elapsed = time.time() - t0
        rate    = (idx + 1) / elapsed
        eta     = (len(test_stems) - idx - 1) / rate / 60
        print(f"  [{idx+1}/{len(test_stems)}]  {elapsed/60:.1f} min elapsed  ETA {eta:.1f} min")

print(f"Inference done in {(time.time()-t0)/60:.1f} min")

sub = (pd.DataFrame(rows)
       .sort_values("image_id")
       .reset_index(drop=True))
sub.to_csv(SUB_PATH, index=False)

n_with = int((sub["PredictionString"].str.len() > 0).sum())
print(f"\nSaved → {SUB_PATH}")
print(f"Rows: {len(sub)} | With detections: {n_with} | Empty: {len(sub)-n_with}")
print(sub.head())
assert len(sub) == len(test_stems), "Row count mismatch!"


2026-06-24 08:31:50,857 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:31:52,047 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:31:53,239 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)


  [1/327]  0.1 min elapsed  ETA 28.2 min


2026-06-24 08:31:54,979 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:31:56,019 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:31:57,075 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:31:58,671 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:31:59,742 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:32:00,804 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:32:02,413 - sahi - WARNING - Switching postprocess type/metric

  [50/327]  3.1 min elapsed  ETA 17.4 min


2026-06-24 08:34:58,609 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:34:59,656 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:35:00,710 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:35:02,317 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:35:03,394 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:35:04,463 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:35:06,111 - sahi - WARNING - Switching postprocess type/metric

  [100/327]  6.3 min elapsed  ETA 14.3 min


... (6 outputs left out)

2026-06-24 08:47:22,186 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:47:23,230 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:47:24,270 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:47:25,801 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:47:26,861 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:47:27,897 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:47:29,425 - sahi - WARNING - Switching postprocess type/metric

  [300/327]  18.6 min elapsed  ETA 1.7 min


2026-06-24 08:50:23,332 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:50:24,383 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:50:25,421 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:50:26,978 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:50:28,034 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:50:29,080 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-06-24 08:50:30,671 - sahi - WARNING - Switching postprocess type/metric

Inference done in 20.2 min

Saved → /kaggle/working/submission_v3.csv
Rows: 327 | With detections: 327 | Empty: 0
                                        image_id  \
0  CCTV01^20260318-110000-20260318-120000_000061   
1  CCTV01^20260318-110000-20260318-120000_000065   
2  CCTV01^20260318-110000-20260318-120000_000069   
3  CCTV01^20260318-110000-20260318-120000_000073   
4  CCTV01^20260318-110000-20260318-120000_000077   

                                    PredictionString  
0  3 77.8322 0.524878 0.878134 0.070234 0.195659 ...  
1  3 78.3344 0.524822 0.877935 0.070018 0.195044 ...  
2  3 72.3498 0.525295 0.878016 0.072198 0.196373 ...  
3  3 77.0523 0.525236 0.878306 0.071661 0.196413 ...  
4  3 78.1338 0.525428 0.879121 0.071153 0.197612 ...  


In [11]:
# ════════════════════════════════════════════════════
#  OPTIONAL — Per-class AP on fold 0 val set
#  Run this to identify which classes are dragging mAP
#  (no retraining needed, just a quick eval)
# ════════════════════════════════════════════════════
print("Running per-class validation on fold 0...")
diag_model = YOLO(str(RUNS_DIR / "fold_0" / "weights" / "best.pt"))
metrics = diag_model.val(
    data=fold_yamls[0],
    imgsz=IMGSZ,
    batch=BATCH,
    conf=0.001,
    iou=0.5,
    device=DEVICE,
    verbose=True,
)

class_names = [str(i) for i in range(NUM_CLASSES)]
print("\nPer-class AP50:")
for i, (name, ap) in enumerate(zip(class_names, metrics.box.ap50)):
    bar = "█" * int(ap * 30)
    print(f"  Class {i:2d} ({name:>2s}): {ap:.4f}  {bar}")
print(f"\nmAP50   : {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print("\nClasses with AP < 0.30 are candidates for extra copy_paste weight or pseudo-labelling.")


Running per-class validation on fold 0...
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 126 layers, 20,040,055 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 478.2±87.5 MB/s, size: 387.8 KB)
val: Scanning /kaggle/temp/roadvision_v3/fold_0/labels/val.cache... 270 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 270/270 125.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 1.5it/s 22.4s
                   all        270       3834      0.476      0.578      0.508      0.262
                     0        167       1027      0.674      0.671      0.682      0.247
                     1        171        538      0.685      0.606      0.649      0.251
                     2        183        689      0.545      0.762      0.603      0.278
                     3        197        629      0.685      0.924      0.8

## Session budget (T4, ~12 hr cap)

| Phase | Est. time |
|---|---|
| VisDrone download + convert | ~5 min |
| VisDrone pretrain (15 epochs @ 640px) | ~1.0 hr |
| Bangladesh fold 0 (80 epochs @ 1280px) | ~2.0 hr |
| Bangladesh fold 1 | ~2.0 hr |
| Bangladesh fold 2 | ~2.0 hr |
| Inference (standard + SAHI + WBF) | ~0.5–1.0 hr |
| **Total** | **~8–9 hr** ✅ |

**If fold 0 exceeds 2.5 hr:** reduce `EPOCHS = 60` and `patience = 15` in Cell 3 and restart from Cell 7.

**If SAHI inference is too slow:** set `USE_SAHI = False` — you still get WBF + TTA from 3 fold models which alone should beat 0.63.

## Expected score
| Change | Est. gain |
|---|---|
| VisDrone pretrain → better generalisation | +0.02–0.04 |
| 80 epochs + richer augmentation | +0.01–0.02 |
| SAHI 512px slices at inference | +0.01–0.02 |
| WBF iou_thr 0.6 → 0.55 | +0.005 |
| Combined | **~0.65–0.70** |
